# Terminations in Microsoft Autogen

## Why Termination Matters

We have used team to do some work, say write a story or accomplish some task. 

In [1]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os

load_dotenv(override=True)

from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("✅ API Key loaded successfully.")
else:
    print("❌ API Key not found.")

✅ API Key loaded successfully.


In [2]:
model_client =  OpenAIChatCompletionClient(
    base_url="https://openrouter.ai/api/v1",
    model="qwen/qwen-2.5-72b-instruct:free",
    api_key = api_key,
    model_info={
        "family":'deepseek',
        "vision" :True,
        "function_calling":True,
        "json_output": False
    }
)

/Users/lokesh/Autogen/venv/lib/python3.13/site-packages/autogen_ext/models/openai/_openai_client.py:439: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


In [3]:
from autogen_agentchat.agents import AssistantAgent
add_1_agent_first = AssistantAgent(
    name = 'add_1_agent_first',
    model_client=model_client,
    system_message="Add 1 to the number, first number is 0. Give result as output"
)

add_1_agent_second = AssistantAgent(
    name = 'add_1_agent_second',
    model_client=model_client,
    system_message="Add 1 to the number. Give result as output."
)
 
add_1_agent_third = AssistantAgent(
    name = 'add_1_agent_third',
    model_client=model_client,
    system_message="Add 1 to the number. Give result as output."
)

from autogen_agentchat.teams import RoundRobinGroupChat

team = RoundRobinGroupChat(
    [add_1_agent_first, add_1_agent_second, add_1_agent_third]
)

In [4]:
from autogen_agentchat.conditions import MaxMessageTermination

max_termination = MaxMessageTermination(5)

In [5]:
team = RoundRobinGroupChat(
    [add_1_agent_first, add_1_agent_second, add_1_agent_third],
    termination_condition =max_termination
)

In [6]:
from autogen_agentchat.ui import Console

await Console(team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
The first number is 0. When you add 1 to it, the result is:

**1**
---------- TextMessage (add_1_agent_second) ----------
The first number is 0. When you add 1 to it, the result is:

**1**
---------- TextMessage (add_1_agent_third) ----------
The first number is 0. When you add 1 to it, the result is:

**1**
---------- TextMessage (add_1_agent_first) ----------
It looks like you're repeating the same instruction. If you add 1 to 0, the result is indeed:

**1**

If you have another number or a different step you'd like to perform, please let me know!
---------- TextMessage (add_1_agent_second) ----------
The first number is 0. When you add 1 to it, the result is:

**1**

If you have another number or a different step you'd like to perform, let me know!


TaskResult(messages=[TextMessage(id='37f33c26-6113-4275-b10d-648a280d408a', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=25, completion_tokens=23), metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 11, 34, 174271, tzinfo=datetime.timezone.utc), content='The first number is 0. When you add 1 to it, the result is:\n\n**1**', type='TextMessage'), TextMessage(id='d849e1f6-61cd-4a9d-8eab-b71017f6a86c', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=53, completion_tokens=23), metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 11, 35, 985775, tzinfo=datetime.timezone.utc), content='The first number is 0. When you add 1 to it, the result is:\n\n**1**', type='TextMessage'), TextMessage(id='c72a1437-805f-4684-9d65-bd5ed9b01f90', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=75, completion_tokens=23), metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 11, 38, 314328, tzinfo=datetime.timezone.utc), content=

In [7]:
from autogen_agentchat.agents import AssistantAgent

In [8]:

agent1 = AssistantAgent(
    name = 'story_writer',
    model_client=model_client,
    system_message="Give the story about a brave knight, keep it short no more than 40 words. if critic say 'THE END' anywhere. Only output 'THE END'"
)

agent2 = AssistantAgent(
    name = 'story_critic',
    model_client=model_client,
    system_message="Continue the story and critic it with feedback. Keep it short and no more than 40 words. If it feels complete, just say 'THE END'. Only output 'THE END'"
)

In [9]:
from autogen_agentchat.conditions import TextMentionTermination


text_mention_termination = TextMentionTermination('THE END')
teamWithTextTermination = RoundRobinGroupChat(
    [agent1, agent2],
    termination_condition=text_mention_termination
)

In [10]:
from autogen_agentchat.ui import Console

await Console(teamWithTextTermination.run_stream(task = 'Write a story about a brave knight.'))

---------- TextMessage (user) ----------
Write a story about a brave knight.
---------- TextMessage (story_writer) ----------
A brave knight, Sir Llion, faced the dragon alone. With his shield and sword, he protected the village, defeating the beast with a single, daring strike. THE END


TaskResult(messages=[TextMessage(id='17cceafc-3fcb-4fb6-8c0a-d36b587b13a0', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 12, 14, 892627, tzinfo=datetime.timezone.utc), content='Write a story about a brave knight.', type='TextMessage'), TextMessage(id='84843ea3-e5ea-4ef8-8b36-9f70ccc75487', source='story_writer', models_usage=RequestUsage(prompt_tokens=57, completion_tokens=37), metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 12, 17, 3306, tzinfo=datetime.timezone.utc), content='A brave knight, Sir Llion, faced the dragon alone. With his shield and sword, he protected the village, defeating the beast with a single, daring strike. THE END', type='TextMessage')], stop_reason="Text 'THE END' mentioned")

# Combining Termination Conditions

In [11]:
combined_termination = MaxMessageTermination(5) | TextMentionTermination('THE END')

from autogen_agentchat.conditions import TextMentionTermination


text_mention_termination = TextMentionTermination('THE END')
teamWithTextTermination = RoundRobinGroupChat(
    [agent1, agent2],
    termination_condition=combined_termination
)

In [12]:
from autogen_agentchat.ui import Console

await Console(teamWithTextTermination.run_stream(task = 'Write a story about a brave knight.'))

---------- TextMessage (user) ----------
Write a story about a brave knight.
---------- TextMessage (story_writer) ----------
THE END


TaskResult(messages=[TextMessage(id='50722709-4baa-4d1e-8d58-c5d68e5ad032', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 12, 55, 484218, tzinfo=datetime.timezone.utc), content='Write a story about a brave knight.', type='TextMessage'), TextMessage(id='eb424a68-6e01-47c5-9f5c-de6199e585bb', source='story_writer', models_usage=RequestUsage(prompt_tokens=113, completion_tokens=3), metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 12, 57, 633465, tzinfo=datetime.timezone.utc), content='THE END', type='TextMessage')], stop_reason="Text 'THE END' mentioned")

# External Termination

ExternalTermination: Enables programmatic control of termination from outside the run. This is useful for UI integration (e.g., “Stop” buttons in chat interfaces).

In [13]:
from autogen_agentchat.agents import AssistantAgent

agent1 = AssistantAgent(
    name = 'story_writer',
    model_client=model_client,
    system_message="Give the story about a brave knight, keep it short no more than 40 words. if critic say 'THE END' anywhere. Only output 'THE END'"
)

agent2 = AssistantAgent(
    name = 'story_critic',
    model_client=model_client,
    system_message="Continue the story and critic it with feedback. Keep it short and no more than 40 words. If it feels complete, just say 'THE END'. Only output 'THE END'"
)

from autogen_agentchat.conditions import ExternalTermination
external_termination = ExternalTermination()

from autogen_agentchat.teams import RoundRobinGroupChat
team = RoundRobinGroupChat(
    [agent1, agent2],
    termination_condition= external_termination
)



In [15]:
import asyncio

In [16]:
from autogen_agentchat.ui import Console
run = asyncio.create_task(Console(team.run_stream(task = 'Write a story about a brave knight less than 40 words.')))

await asyncio.sleep(10)

external_termination.set()
await run

---------- TextMessage (user) ----------
Write a story about a brave knight less than 40 words.
---------- TextMessage (story_writer) ----------
A brave knight faced a fearsome dragon, sword in hand, to save his village. He struck the final blow, triumphing against all odds. THE END
---------- TextMessage (story_critic) ----------
THE END

Feedback: The story is concise and captures the essence of bravery and triumph. It could benefit from a bit more detail or a twist to make it stand out.
---------- TextMessage (story_writer) ----------
THE END
---------- TextMessage (story_critic) ----------
THE END

Feedback: The story is clear and direct, effectively conveying the knight's bravery and victory. Adding a unique element or internal struggle could deepen the narrative.
---------- TextMessage (story_writer) ----------
THE END


TaskResult(messages=[TextMessage(id='dfbacf40-27c1-4741-adec-496ee37f860a', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 14, 14, 982930, tzinfo=datetime.timezone.utc), content='Write a story about a brave knight less than 40 words.', type='TextMessage'), TextMessage(id='f1b9e721-150f-46c1-97a7-dc2be7e58d22', source='story_writer', models_usage=RequestUsage(prompt_tokens=63, completion_tokens=33), metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 14, 17, 693395, tzinfo=datetime.timezone.utc), content='A brave knight faced a fearsome dragon, sword in hand, to save his village. He struck the final blow, triumphing against all odds. THE END', type='TextMessage'), TextMessage(id='56dc4859-3d09-4bf1-abb4-b56b0d9e1af4', source='story_critic', models_usage=RequestUsage(prompt_tokens=99, completion_tokens=36), metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 14, 19, 721673, tzinfo=datetime.timezone.utc), content='THE END\n\nFeed

In [ ]:
# The team is not stopping immediately but rather current agent is completing its run.

# Aborting A Team

Different from stopping a team, aborting a team will immediately stop the team and raise a CancelledError exception.

In [17]:
from autogen_core import CancellationToken

cancellation_token = CancellationToken()

run2 = asyncio.create_task(
    Console(team.run_stream(task = 'Give a short Story about a lion atmost 40 words',cancellation_token=cancellation_token))
)

await asyncio.sleep(5)
cancellation_token.cancel()

try:
    result = await run2
except :
    print("Task Was Cancelled")

---------- TextMessage (user) ----------
Give a short Story about a lion atmost 40 words
---------- TextMessage (story_critic) ----------
A lion, king of the jungle, roared mightily, asserting his dominance. Yet, he showed mercy to the weak, proving true strength lies in compassion. THE END
---------- TextMessage (story_writer) ----------
THE END


Error processing publish message for story_critic_cb133f8e-2f4c-4b4d-90b1-13929871ac7e/cb133f8e-2f4c-4b4d-90b1-13929871ac7e
Traceback (most recent call last):
  File "/Users/lokesh/Autogen/venv/lib/python3.13/site-packages/autogen_core/_single_threaded_agent_runtime.py", line 606, in _on_message
    return await agent.on_message(
           ^^^^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
    )
    ^
  File "/Users/lokesh/Autogen/venv/lib/python3.13/site-packages/autogen_core/_base_agent.py", line 119, in on_message
    return await self.on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/lokesh/Autogen/venv/lib/python3.13/site-packages/autogen_agentchat/teams/_group_chat/_sequential_routed_agent.py", line 67, in on_message_impl
    return await super().on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/lokesh/Autogen/venv/lib/python3.13/site-packages/autogen_core/_routed_agent.py", line 485, in on

Task Was Cancelled
